# 05 — Recommendation System

**Project:** Credit Risk Assessment — *Give Me Some Credit* (Kaggle)

Notebooks 01–04 answered *can we predict default, and can we trust the score?*
`04_evaluation` selected a **calibrated XGBoost** operating at the cost-optimal threshold
$p^{*} \approx 0.167$ and persisted the model with its decision policy. This notebook turns
that artifact into a usable **recommendation system**: a function that takes a *raw*
applicant record — exactly as a loan officer would type it — and returns the three things a
real lending decision requires:

1. a **calibrated probability of default** $\hat{p}$,
2. a **decision** (approve / refer to manual review / decline) under an explicit cost policy,
3. the **reason codes** — the signed, per-applicant factors behind the score, via SHAP.

The third item is not optional polish. Fair-lending regulation (e.g. the U.S. Equal Credit
Opportunity Act, and the GDPR "right to an explanation" in the EU) requires that a declined
applicant be told *why*. A credit model that cannot explain itself is not deployable.

### The inference contract and training/serving skew

The model was trained on *processed* features (cleaned, winsorised, imputed, plus
missingness flags). A new applicant arrives as **raw** values. If the transformation at
inference differs in the slightest from the one at training — a different median, a
different cap, a missed cleaning rule — the model silently receives out-of-distribution
inputs. This failure mode is **training/serving skew**, one of the most common causes of
models that look excellent offline yet misbehave in production.

Notebook 02 persisted the transformed datasets but not a reusable transformer. We therefore
**reconstruct the preprocessing as a single fitted object**, learning its parameters from
the *same* deterministic training split (`random_state=42`), and we **prove faithfulness**
by checking that re-processing the raw test rows reproduces the saved `X_test.csv` exactly.
Only a transformer that passes this test is safe to deploy.

In [1]:
import sys

!{sys.executable} -m pip install xgboost shap --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET = "SeriousDlqin2yrs"
TEST_SIZE = 0.20

DATA_RAW   = Path("..") / "data" / "cs-training.csv"
PROC_DIR   = Path("data") / "processed"
MODELS_DIR = Path("models")

FINAL_MODEL_PATH = MODELS_DIR / "final_model.joblib"
POLICY_PATH      = MODELS_DIR / "decision_policy.json"
XGB_TUNED_PATH   = MODELS_DIR / "xgboost.joblib"

RAW_FEATURES = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
]

MISSING_FLAG_COLS = ["MonthlyIncome", "NumberOfDependents"]

PASTDUE_COLS = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

pd.set_option("display.max_columns", None)
print("shap version:", shap.__version__)

shap version: 0.52.0


In [4]:
final_model = joblib.load(FINAL_MODEL_PATH)    
xgb_tuned   = joblib.load(XGB_TUNED_PATH)      

with open(POLICY_PATH, "r", encoding="utf-8") as fh:
    policy = json.load(fh)

DECISION_THRESHOLD = float(policy["threshold"])        
C_FN = float(policy["cost_false_negative"])              
C_FP = float(policy["cost_false_positive"])              

X_test_stored = pd.read_csv(PROC_DIR / "X_test.csv")
STORED_ORDER  = X_test_stored.columns.tolist()          

print(f"Final model        : {type(final_model).__name__}")
print(f"Base model (SHAP)  : {type(xgb_tuned).__name__}")
print(f"Decision policy    : decline-leaning if P(default) > {DECISION_THRESHOLD:.3f}")
print(f"Cost ratio C_FN:C_FP = {C_FN:.0f}:{C_FP:.0f}")
print(f"Stored feature order ({len(STORED_ORDER)}): {STORED_ORDER}")

Final model        : CalibratedClassifierCV
Base model (SHAP)  : XGBClassifier
Decision policy    : decline-leaning if P(default) > 0.167
Cost ratio C_FN:C_FP = 5:1
Stored feature order (12): ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'MonthlyIncome_missing', 'NumberOfDependents_missing']


## 1. The inference pipeline

A deployed prediction is a **composition** of two stages:

$$
\hat{p}(\mathbf{x}_{\text{raw}}) \;=\; \hat{f}_{\text{cal}}\big(T(\mathbf{x}_{\text{raw}})\big),
$$

where $T$ is the preprocessing transform (cleaning, winsorisation, imputation, missingness
flags) and $\hat{f}_{\text{cal}}$ is the calibrated XGBoost from notebook 04. For the result
to be valid, $T$ at serving time must be **identical** to $T$ at training time — same caps,
same medians, same rules. Any divergence is training/serving skew.

$T$ has two kinds of parameters:
- **structural** rules that never change (which codes are junk, which columns get a flag);
- **learned** parameters — the winsor caps and imputation medians — which must be estimated
  on the **training split only**, never on the data we score, to avoid leakage.

The split in `02_preprocessing` is deterministic (`train_test_split(..., random_state=42,
stratify=y)`), so re-running it here recovers the *exact same* raw training rows. We fit $T$
on them, then validate $T$ by reproducing the stored test matrix below.

In [5]:
class CreditPreprocessor:
    """Leakage-safe reconstruction of the 02_preprocessing transform.

    Mirrors notebook 02 exactly: missingness flags on the original values; junk
    codes (96/98) and the impossible age == 0 set to NaN; the two heavy-tailed
    ratios winsorised at train-derived 99th-percentile caps; then median
    imputation of every column that can hold NaN. All learned parameters come
    from the RAW training split only.
    """

    WINSOR_COLS = ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]
    WINSOR_Q = 0.99
    PLACEHOLDERS = [96, 98]                       

    IMPUTE_COLS = ["age", "MonthlyIncome", "NumberOfDependents"] + PASTDUE_COLS

    def __init__(self, stored_order):
        self.stored_order = stored_order
        self.caps_ = {}
        self.medians_ = {}

    def _structural_clean(self, X):
        """Parameter-free rules (02 §4): junk codes and age == 0 become NaN."""
        X = X.copy()
        for c in PASTDUE_COLS:
            X[c] = X[c].mask(X[c].isin(self.PLACEHOLDERS))
        X["age"] = X["age"].mask(X["age"] == 0)
        return X

    def fit(self, X_raw):
        X = self._structural_clean(X_raw[RAW_FEATURES])
        for c in self.WINSOR_COLS:
            self.caps_[c] = float(X[c].quantile(self.WINSOR_Q))
            X[c] = X[c].clip(upper=self.caps_[c])
        for c in self.IMPUTE_COLS:
            self.medians_[c] = float(X[c].median())
        return self

    def transform(self, X_raw):
        X = X_raw[RAW_FEATURES].copy()
        flags = {f"{c}_missing": X[c].isna().astype(int) for c in MISSING_FLAG_COLS}
        X = self._structural_clean(X)
        for c in self.WINSOR_COLS:
            X[c] = X[c].clip(upper=self.caps_[c])
        for c in self.IMPUTE_COLS:
            X[c] = X[c].fillna(self.medians_[c])
        for name, col in flags.items():
            X[name] = col.values
        return X[self.stored_order]

## 2. Proving the transform is faithful

A reconstructed transform is only trustworthy if it reproduces the original. We re-run the
deterministic split to recover the raw test rows, fit the preprocessor on the raw *training*
rows, transform the raw test rows, and assert the result equals the stored `X_test.csv`
**exactly**. If a single cap, median or rule diverged, the assertion fails — turning a silent
production bug into a loud, immediate test failure. This is the engineering guarantee that
makes the system safe to deploy.

In [7]:
raw = pd.read_csv(DATA_RAW)
if "Unnamed: 0" in raw.columns:                 
    raw = raw.drop(columns=["Unnamed: 0"])

X_raw_all, y_all = raw[RAW_FEATURES], raw[TARGET]

X_raw_train, X_raw_test, _, _ = train_test_split(
    X_raw_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)


preprocessor = CreditPreprocessor(stored_order=STORED_ORDER).fit(X_raw_train)
X_test_rebuilt = preprocessor.transform(X_raw_test).reset_index(drop=True)

pd.testing.assert_frame_equal(
    X_test_rebuilt,
    X_test_stored.reset_index(drop=True),
    check_dtype=False,
    atol=1e-6,
)
print(" Faithfulness check passed: the reconstructed transform reproduces X_test.csv.")
print(f"  Learned winsor caps : {preprocessor.caps_}")
print(f"  Learned medians     : {preprocessor.medians_}")

 Faithfulness check passed: the reconstructed transform reproduces X_test.csv.
  Learned winsor caps : {'RevolvingUtilizationOfUnsecuredLines': 1.0944386634299974, 'DebtRatio': 4977.0}
  Learned medians     : {'age': 52.0, 'MonthlyIncome': 5390.0, 'NumberOfDependents': 0.0, 'NumberOfTime30-59DaysPastDueNotWorse': 0.0, 'NumberOfTime60-89DaysPastDueNotWorse': 0.0, 'NumberOfTimes90DaysLate': 0.0}


## 3. The decision function

With $T$ validated, the deployed score is the composition from §1, evaluated with the
**calibrated** model so the probability can be read literally:

$$
\hat{p}(\mathbf{x}) = \hat{f}_{\text{cal}}\big(T(\mathbf{x})\big),
\qquad
\text{decision}(\mathbf{x}) = \mathbb{1}\!\left[\hat{p}(\mathbf{x}) \ge p^{*}\right].
$$

Calibration is what makes this meaningful: because notebook 04 calibrated the scores, a
returned $\hat{p}=0.21$ genuinely means *about a 21% chance of default*. That is exactly what
lets us compare it to the cost-derived threshold $p^{*} = C_{FP}/(C_{FN}+C_{FP})$, and what
lets a human reason about the number at all. On raw, miscalibrated scores the comparison
$\hat{p} \ge p^{*}$ would be meaningless.

`predict_risk` below is the thin, side-effect-free **core** of the system: a raw record in, a
probability and a decision out. Everything downstream (the policy, reason codes, the report)
builds on this one function.

In [8]:
def predict_risk(applicant: dict, threshold: float = DECISION_THRESHOLD) -> dict:
    """Score one raw applicant; return a calibrated probability and a binary decision.

    Args:
        applicant: mapping from each name in RAW_FEATURES to its raw value.
            Missing income or dependents may be passed as None / np.nan.
        threshold: decision threshold; defaults to the cost-optimal p* from notebook 04.

    Returns:
        dict with keys 'probability' (float in [0, 1]) and 'decision'
        ('DECLINE' or 'APPROVE').
    """
    missing = set(RAW_FEATURES) - set(applicant)
    if missing:
        raise ValueError(f"Missing required features: {sorted(missing)}")

    x_raw = pd.DataFrame([{k: applicant[k] for k in RAW_FEATURES}], columns=RAW_FEATURES)
    z = preprocessor.transform(x_raw)                       # same T as training
    prob = float(final_model.predict_proba(z)[0, 1])        # calibrated P(default)
    return {"probability": prob, "decision": "DECLINE" if prob >= threshold else "APPROVE"}


# Smoke test on the first stored test applicant.
_demo = predict_risk({c: X_raw_test.iloc[0][c] for c in RAW_FEATURES})
print(_demo)

{'probability': 0.007064760196954012, 'decision': 'APPROVE'}


## 4. From a single threshold to a three-tier policy

A pure two-way rule (approve / decline at $p^{*}$) is statistically optimal but operationally
blunt: applicants whose probability sits *near* $p^{*}$ are exactly the ones the model is
least sure about, and a small error in $\hat{p}$ flips the decision. Real lenders add a
**manual-review band** around the threshold and send borderline cases to a human:

$$
\text{decision}(p) =
\begin{cases}
\text{APPROVE} & p < p^{*} \\
\text{MANUAL REVIEW} & p^{*} \le p < p_{\text{decline}} \\
\text{DECLINE} & p \ge p_{\text{decline}}
\end{cases}
$$

Here $p^{*}$ comes from the **cost mathematics** of notebook 04; $p_{\text{decline}}$ is an
explicit **business** parameter (we use $2p^{*}$), declared as such. Setting
$p_{\text{decline}} = p^{*}$ recovers the pure two-way rule. We also expose the two expected
costs $\mathbb{E}[\text{cost}\mid\text{approve}] = p\,C_{FN}$ and
$\mathbb{E}[\text{cost}\mid\text{decline}] = (1-p)\,C_{FP}$, so a reviewer can see the
trade-off behind any borderline case.

In [9]:
@dataclass
class DecisionPolicy:
    """Three-tier lending policy over a calibrated probability of default."""
    threshold: float           # p* : approve strictly below this
    decline_threshold: float   # decline at or above this; review in between
    cost_fn: float
    cost_fp: float

    def decide(self, p: float) -> str:
        if p < self.threshold:
            return "APPROVE"
        if p < self.decline_threshold:
            return "MANUAL REVIEW"
        return "DECLINE"

    def expected_costs(self, p: float) -> dict:
        """Expected unit cost of each hard action at probability p."""
        return {"approve": p * self.cost_fn, "decline": (1 - p) * self.cost_fp}


policy_obj = DecisionPolicy(
    threshold=DECISION_THRESHOLD,
    decline_threshold=2 * DECISION_THRESHOLD,   # business parameter (review band width)
    cost_fn=C_FN,
    cost_fp=C_FP,
)
print(f"APPROVE  if p <  {policy_obj.threshold:.3f}")
print(f"REVIEW   if      {policy_obj.threshold:.3f} <= p < {policy_obj.decline_threshold:.3f}")
print(f"DECLINE  if p >= {policy_obj.decline_threshold:.3f}")

APPROVE  if p <  0.167
REVIEW   if      0.167 <= p < 0.333
DECLINE  if p >= 0.333


## 5. Reason codes via SHAP

A probability tells an applicant *what* the model decided, not *why*. From notebook 04, the
SHAP value $\phi_i$ is feature $i$'s additive contribution to the model's log-odds margin,
with the local-accuracy guarantee

$$
f(\mathbf{x}) = \phi_0 + \sum_{i} \phi_i, \qquad \phi_0 = \mathbb{E}[f(X)].
$$

A **positive** $\phi_i$ pushes the default log-odds — and hence the probability — **up** (a
risk factor); a **negative** $\phi_i$ pushes it down (a protective factor). We surface the
few features with the largest $|\phi_i|$, tagged by sign, as the applicant's reason codes.

**Why explain the base XGBoost, not the calibrated wrapper?** TreeSHAP needs a tree model;
`CalibratedClassifierCV` is a monotone post-processor wrapping cloned trees, not itself a
tree. Because calibration is **monotone**, it never changes the *ranking* or the *sign* of a
feature's effect — only the numeric probability. So SHAP on the base XGBoost gives the
correct *direction and relative magnitude* of each reason code, while the calibrated model
supplies the trustworthy probability. The two roles are complementary.

In [10]:
explainer = shap.TreeExplainer(xgb_tuned)

# Human-readable labels for an applicant-facing explanation.
FEATURE_LABELS = {
    "RevolvingUtilizationOfUnsecuredLines": "Credit utilisation ratio",
    "age": "Age",
    "NumberOfTime30-59DaysPastDueNotWorse": "Times 30–59 days past due",
    "DebtRatio": "Debt-to-income ratio",
    "MonthlyIncome": "Monthly income",
    "NumberOfOpenCreditLinesAndLoans": "Open credit lines & loans",
    "NumberOfTimes90DaysLate": "Times 90+ days late",
    "NumberRealEstateLoansOrLines": "Real-estate loans / lines",
    "NumberOfTime60-89DaysPastDueNotWorse": "Times 60–89 days past due",
    "NumberOfDependents": "Number of dependents",
    "MonthlyIncome_missing": "Monthly income not provided",
    "NumberOfDependents_missing": "Dependents not provided",
}


def reason_codes(applicant: dict, top_n: int = 4) -> pd.DataFrame:
    """Top-n signed SHAP factors behind one applicant's score (in log-odds)."""
    z = preprocessor.transform(
        pd.DataFrame([{k: applicant[k] for k in RAW_FEATURES}], columns=RAW_FEATURES)
    )
    sv = explainer(z)
    contrib = pd.DataFrame({
        "feature": [FEATURE_LABELS.get(c, c) for c in z.columns],
        "value": z.iloc[0].values,
        "shap": sv.values[0],
    })
    contrib["effect"] = np.where(contrib["shap"] >= 0, "↑ increases risk", "↓ lowers risk")
    return (contrib.reindex(contrib["shap"].abs().sort_values(ascending=False).index)
                   .head(top_n)
                   .reset_index(drop=True))


print(reason_codes({c: X_raw_test.iloc[0][c] for c in RAW_FEATURES}))

                     feature      value      shap         effect
0   Credit utilisation ratio   0.019252 -1.126539  ↓ lowers risk
1                        Age  66.000000 -0.388477  ↓ lowers risk
2  Times 30–59 days past due   0.000000 -0.327738  ↓ lowers risk
3        Times 90+ days late   0.000000 -0.265168  ↓ lowers risk


## 6. The end-to-end recommendation

We compose the three pieces — calibrated probability, three-tier policy, and SHAP reason
codes — into a single applicant-facing report. This is the artifact a loan officer would
read: a decision, the probability behind it, the expected costs of the hard alternatives,
and the specific factors driving the score (the adverse-action explanation).

In [12]:
def recommend(applicant: dict, top_n: int = 4) -> dict:
    """Full recommendation: probability, decision, expected costs, and reason codes."""
    risk = predict_risk(applicant)
    p = risk["probability"]
    return {
        "probability": p,
        "decision": policy_obj.decide(p),
        "expected_costs": policy_obj.expected_costs(p),
        "reason_codes": reason_codes(applicant, top_n=top_n),
    }


def print_recommendation(applicant: dict) -> None:
    rec = recommend(applicant)
    ec = rec["expected_costs"]
    print("=" * 56)
    print(f"  DECISION         : {rec['decision']}")
    print(f"  P(default)       : {rec['probability']:.1%}")
    print(f"  Threshold p*     : {DECISION_THRESHOLD:.1%}")
    print(f"  Expected cost    : approve={ec['approve']:.3f}  decline={ec['decline']:.3f}")
    print("  Top reason codes :")
    for _, r in rec["reason_codes"].iterrows():
        print(f"    {r['effect']:<18}  {r['feature']} (= {r['value']:.2f})")
    print("=" * 56)

In [13]:
# Three illustrative applicants spanning the decision tiers.
low_risk = {
    "RevolvingUtilizationOfUnsecuredLines": 0.05, "age": 52,
    "NumberOfTime30-59DaysPastDueNotWorse": 0, "DebtRatio": 0.20,
    "MonthlyIncome": 9000, "NumberOfOpenCreditLinesAndLoans": 6,
    "NumberOfTimes90DaysLate": 0, "NumberRealEstateLoansOrLines": 1,
    "NumberOfTime60-89DaysPastDueNotWorse": 0, "NumberOfDependents": 0,
}
high_risk = {
    "RevolvingUtilizationOfUnsecuredLines": 0.95, "age": 27,
    "NumberOfTime30-59DaysPastDueNotWorse": 3, "DebtRatio": 0.85,
    "MonthlyIncome": 2200, "NumberOfOpenCreditLinesAndLoans": 4,
    "NumberOfTimes90DaysLate": 2, "NumberRealEstateLoansOrLines": 0,
    "NumberOfTime60-89DaysPastDueNotWorse": 1, "NumberOfDependents": 2,
}
missing_income = {**low_risk, "MonthlyIncome": np.nan, "NumberOfTime30-59DaysPastDueNotWorse": 1}

for name, applicant in [("LOW RISK", low_risk),
                        ("HIGH RISK", high_risk),
                        ("MISSING INCOME", missing_income)]:
    print(f"\n### {name}")
    print_recommendation(applicant)


### LOW RISK
  DECISION         : APPROVE
  P(default)       : 0.7%
  Threshold p*     : 16.7%
  Expected cost    : approve=0.034  decline=0.993
  Top reason codes :
    ↓ lowers risk       Credit utilisation ratio (= 0.05)
    ↓ lowers risk       Times 30–59 days past due (= 0.00)
    ↓ lowers risk       Times 90+ days late (= 0.00)
    ↓ lowers risk       Debt-to-income ratio (= 0.20)

### HIGH RISK
  DECISION         : DECLINE
  P(default)       : 67.8%
  Threshold p*     : 16.7%
  Expected cost    : approve=3.392  decline=0.322
  Top reason codes :
    ↑ increases risk    Times 90+ days late (= 2.00)
    ↑ increases risk    Times 30–59 days past due (= 3.00)
    ↑ increases risk    Times 60–89 days past due (= 1.00)
    ↑ increases risk    Credit utilisation ratio (= 0.95)

### MISSING INCOME
  DECISION         : APPROVE
  P(default)       : 2.9%
  Threshold p*     : 16.7%
  Expected cost    : approve=0.146  decline=0.971
  Top reason codes :
    ↓ lowers risk       Credit utilisa

## 7. (Optional) From probability to a credit score — a points-based scorecard

Lenders rarely expose a raw probability; they expose a **score** (à la FICO) on which
"points to double the odds" (PDO) is constant. The standard scaling is linear in the
log-odds. Define the good:bad odds $\text{odds} = (1-p)/p$ and set

$$
\text{Score} = \text{Offset} + \text{Factor}\cdot \ln(\text{odds}),
\qquad
\text{Factor} = \frac{\text{PDO}}{\ln 2},
\qquad
\text{Offset} = S_0 - \text{Factor}\cdot\ln(\text{odds}_0).
$$

With a base score $S_0 = 600$ at base odds $\text{odds}_0 = 50{:}1$ and $\text{PDO}=20$,
the odds double every 20 points — higher score, lower risk. Because the map is a strictly
increasing function of $\ln(\text{odds})$, it preserves the model's ranking and the
decision threshold maps to a single **cut-off score**.

In [14]:
PDO, BASE_SCORE, BASE_ODDS = 20.0, 600.0, 50.0
FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)


def prob_to_score(p: float) -> int:
    """Map a calibrated default probability to a points-based credit score."""
    p = min(max(p, 1e-6), 1 - 1e-6)             # guard the log at the extremes
    odds = (1 - p) / p                           # good : bad
    return int(round(OFFSET + FACTOR * np.log(odds)))


cutoff_score = prob_to_score(DECISION_THRESHOLD)
print(f"Decision cut-off score (p* = {DECISION_THRESHOLD:.3f}): {cutoff_score}\n")
for name, applicant in [("LOW RISK", low_risk), ("HIGH RISK", high_risk)]:
    p = predict_risk(applicant)["probability"]
    print(f"{name:<10}  P(default)={p:.1%}  score={prob_to_score(p)}")

Decision cut-off score (p* = 0.167): 534

LOW RISK    P(default)=0.7%  score=631
HIGH RISK   P(default)=67.8%  score=466


## 8. Conclusion

This notebook closed the loop from a trained model to a deployable recommendation system.
We reconstructed the `02_preprocessing` transform as a single fitted, reusable object and
**proved** — via an exact reproduction of the frozen test set — that it is free of
training/serving skew. On top of it we built a side-effect-free scoring core, a transparent
three-tier decision policy separating cost-optimal mathematics from explicit business
parameters, and SHAP-based reason codes that give every decision an auditable, signed
explanation. An optional points-based scorecard maps the calibrated probability onto the
score format lenders actually use.

The result is a system that is **correct** (validated against the held-out data),
**explainable** (per-applicant reason codes), and **honest** about which of its parameters
are statistical and which are business choices — the three properties a deployed credit
model is judged on.

### References
- Lundberg & Lee (2017) — *A Unified Approach to Interpreting Model Predictions* (SHAP).
- Lundberg et al. (2018) — *Consistent Individualized Feature Attribution for Tree Ensembles* (TreeSHAP).
- Siddiqi (2017) — *Intelligent Credit Scoring* (scorecard / PDO scaling).
- Sculley et al. (2015) — *Hidden Technical Debt in Machine Learning Systems* (training/serving skew).
- scikit-learn, XGBoost and SHAP official documentation.